# Lab 4: Mini Transformer NMT

In [1]:
import math
import random
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## TODO: Implement Mini Transformer

## 1) Build vocabulary and encode sentences
We load EN-VI parallel data, build token vocabularies with special tokens, and convert text into token IDs.

In [2]:
PAD_TOKEN = "<pad>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

def tokenize(text: str):
    return text.strip().lower().split()

def read_parallel(src_path: Path, tgt_path: Path, max_samples: int | None = None):
    with src_path.open("r", encoding="utf-8") as fs, tgt_path.open("r", encoding="utf-8") as ft:
        src_lines = [line.strip() for line in fs if line.strip()]
        tgt_lines = [line.strip() for line in ft if line.strip()]

    n = min(len(src_lines), len(tgt_lines))
    pairs = list(zip(src_lines[:n], tgt_lines[:n]))
    if max_samples is not None:
        pairs = pairs[:max_samples]
    return pairs

class Vocab:
    def __init__(self, tokens_list, min_freq=1):
        counter = Counter()
        for tokens in tokens_list:
            counter.update(tokens)

        self.itos = list(SPECIAL_TOKENS)
        for tok, freq in counter.items():
            if freq >= min_freq and tok not in self.itos:
                self.itos.append(tok)

        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.pad_id = self.stoi[PAD_TOKEN]
        self.bos_id = self.stoi[BOS_TOKEN]
        self.eos_id = self.stoi[EOS_TOKEN]
        self.unk_id = self.stoi[UNK_TOKEN]

    def encode(self, tokens):
        return [self.stoi.get(t, self.unk_id) for t in tokens]

    def decode(self, ids, skip_special=True):
        toks = []
        for idx in ids:
            tok = self.itos[idx]
            if skip_special and tok in SPECIAL_TOKENS:
                continue
            toks.append(tok)
        return toks

    def __len__(self):
        return len(self.itos)

data_dir = Path("data")
train_pairs = read_parallel(data_dir / "train.en", data_dir / "train.vi", max_samples=12000)
dev_pairs = read_parallel(data_dir / "dev.en", data_dir / "dev.vi", max_samples=1500)

train_src_tokens = [tokenize(s) for s, _ in train_pairs]
train_tgt_tokens = [tokenize(t) for _, t in train_pairs]

min_freq = 1 if len(train_pairs) < 1000 else 2
src_vocab = Vocab(train_src_tokens, min_freq=min_freq)
tgt_vocab = Vocab(train_tgt_tokens, min_freq=min_freq)

print(f"Train pairs: {len(train_pairs)} | Dev pairs: {len(dev_pairs)}")
print(f"Source vocab size: {len(src_vocab)}")
print(f"Target vocab size: {len(tgt_vocab)}")
print(f"Using min_freq={min_freq}")

Train pairs: 10 | Dev pairs: 3
Source vocab size: 35
Target vocab size: 44
Using min_freq=1


In [3]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab: Vocab, tgt_vocab: Vocab):
        self.examples = []
        for src_text, tgt_text in pairs:
            src_ids = [src_vocab.bos_id] + src_vocab.encode(tokenize(src_text)) + [src_vocab.eos_id]
            tgt_ids = [tgt_vocab.bos_id] + tgt_vocab.encode(tokenize(tgt_text)) + [tgt_vocab.eos_id]
            self.examples.append((torch.tensor(src_ids), torch.tensor(tgt_ids)))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate_batch(batch, pad_id_src, pad_id_tgt):
    src_list, tgt_list = zip(*batch)
    src_batch = pad_sequence(src_list, padding_value=pad_id_src)
    tgt_batch = pad_sequence(tgt_list, padding_value=pad_id_tgt)
    return src_batch, tgt_batch

train_dataset = TranslationDataset(train_pairs, src_vocab, tgt_vocab)
dev_dataset = TranslationDataset(dev_pairs, src_vocab, tgt_vocab)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: collate_batch(b, src_vocab.pad_id, tgt_vocab.pad_id),
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: collate_batch(b, src_vocab.pad_id, tgt_vocab.pad_id),
)

src_sample, tgt_sample = next(iter(train_loader))
print("src batch shape [S, B]:", tuple(src_sample.shape))
print("tgt batch shape [T, B]:", tuple(tgt_sample.shape))

src batch shape [S, B]: (8, 10)
tgt batch shape [T, B]: (11, 10)


## 2) Positional encoding + Mini Transformer (`nn.Transformer`)

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(0)]
        return self.dropout(x)

class MiniTransformerNMT(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int = 256,
        nhead: int = 8,
        num_encoder_layers: int = 3,
        num_decoder_layers: int = 3,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=False,
        )
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def _generate_square_subsequent_mask(self, sz: int, device):
        return torch.triu(torch.ones((sz, sz), dtype=torch.bool, device=device), diagonal=1)

    def forward(self, src, tgt_in, src_pad_mask, tgt_pad_mask):
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt_in) * math.sqrt(self.d_model))

        tgt_mask = self._generate_square_subsequent_mask(tgt_in.size(0), tgt_in.device)

        memory = self.transformer.encoder(
            src_emb,
            src_key_padding_mask=src_pad_mask,
        )
        out = self.transformer.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )
        logits = self.generator(out)
        return logits

model = MiniTransformerNMT(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512,
    dropout=0.1,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4, betas=(0.9, 0.98), eps=1e-9)

sum_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {sum_params:,}")

d:\hands-on-projects\.venv\Lib\site-packages\torch\nn\modules\transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


Trainable params: 3,986,220


## 3) Train the model

In [5]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.set_grad_enabled(is_train):
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_in = tgt[:-1, :]
            tgt_out = tgt[1:, :]

            src_pad_mask = (src.transpose(0, 1) == src_vocab.pad_id)
            tgt_pad_mask = (tgt_in.transpose(0, 1) == tgt_vocab.pad_id)

            logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            non_pad = (tgt_out != tgt_vocab.pad_id).sum().item()
            total_loss += loss.item() * non_pad
            total_tokens += non_pad

    return total_loss / max(total_tokens, 1)

NUM_EPOCHS = 30
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss = run_epoch(model, dev_loader, optimizer=None)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

Epoch 01 | train_loss=4.0611 | val_loss=3.2936
Epoch 02 | train_loss=3.6028 | val_loss=3.0433
Epoch 03 | train_loss=3.4175 | val_loss=2.8771
Epoch 04 | train_loss=3.2620 | val_loss=2.7226
Epoch 05 | train_loss=3.1125 | val_loss=2.5740
Epoch 06 | train_loss=2.9400 | val_loss=2.4052
Epoch 07 | train_loss=2.8240 | val_loss=2.2445
Epoch 08 | train_loss=2.6474 | val_loss=2.0944
Epoch 09 | train_loss=2.5783 | val_loss=1.9636
Epoch 10 | train_loss=2.3883 | val_loss=1.8398
Epoch 11 | train_loss=2.3345 | val_loss=1.7334
Epoch 12 | train_loss=2.2498 | val_loss=1.6281
Epoch 13 | train_loss=2.0780 | val_loss=1.5387
Epoch 14 | train_loss=1.9956 | val_loss=1.4629
Epoch 15 | train_loss=1.8776 | val_loss=1.3944
Epoch 16 | train_loss=1.8558 | val_loss=1.3236
Epoch 17 | train_loss=1.7348 | val_loss=1.2533
Epoch 18 | train_loss=1.6791 | val_loss=1.1822
Epoch 19 | train_loss=1.5933 | val_loss=1.0961
Epoch 20 | train_loss=1.4988 | val_loss=1.0200
Epoch 21 | train_loss=1.5208 | val_loss=0.9505
Epoch 22 | tr

## 4) Greedy decoding and sample translations

In [6]:
def greedy_decode(model, src_ids, max_len=50):
    model.eval()
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(1)
    src_pad_mask = (src.transpose(0, 1) == src_vocab.pad_id)

    generated = [tgt_vocab.bos_id]
    with torch.no_grad():
        for _ in range(max_len):
            tgt_in = torch.tensor(generated, dtype=torch.long, device=device).unsqueeze(1)
            tgt_pad_mask = (tgt_in.transpose(0, 1) == tgt_vocab.pad_id)

            logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask)
            next_id = logits[-1, 0].argmax(dim=-1).item()
            generated.append(next_id)

            if next_id == tgt_vocab.eos_id:
                break

    return generated

def translate_sentence(sentence: str):
    src_ids = [src_vocab.bos_id] + src_vocab.encode(tokenize(sentence)) + [src_vocab.eos_id]
    pred_ids = greedy_decode(model, src_ids)
    pred_tokens = tgt_vocab.decode(pred_ids, skip_special=True)
    return " ".join(pred_tokens)

for i in range(min(5, len(dev_pairs))):
    src_text, tgt_text = dev_pairs[i]
    pred_text = translate_sentence(src_text)
    print(f"EN  : {src_text}")
    print(f"VI* : {pred_text}")
    print(f"VI  : {tgt_text}")
    print("-" * 80)

EN  : i am a teacher
VI* : tôi là giáo viên
VI  : tôi là giáo viên
--------------------------------------------------------------------------------
EN  : she likes music
VI* : cô ấy thích âm nhạc
VI  : cô ấy thích âm nhạc
--------------------------------------------------------------------------------
EN  : this is a book
VI* : đây là một cuốn sách
VI  : đây là một cuốn sách
--------------------------------------------------------------------------------
